# 01 — The labels, and what the task actually is

Notebook 00 was about the catalogs. This one is about the **six numbers attached to each
catalog** — what they mean, how they were chosen, how to load them, and what they do to a
simulation.

By the end you will know:

1. what each of the six parameters controls,
2. how the 1000 simulations per suite sample them, and where a suite breaks the pattern,
3. how to attach a label to a catalog without getting the rows out of step,
4. what a few crude summary numbers can already see,
5. and the exact task: what you predict, how you are split, how you are scored.

In [ ]:
# Setup. On Colab this installs the toolkit, mounts the data bucket and points the
# environment variables at it. Anywhere else -- a cluster with the data already on disk --
# it does nothing, which is why there is one set of notebooks rather than two.
import sys

if "google.colab" in sys.modules:
    # --force-reinstall, every time, on purpose. Installing only when the package is
    # missing means anyone who ran a notebook once keeps a stale copy forever, and during
    # an event where fixes are being pushed that is exactly backwards. --no-deps keeps it
    # to a few seconds: everything it depends on is already in the runtime.
    %pip install -q --upgrade --force-reinstall --no-deps git+https://github.com/xwzhang98/kaai-robust-inference-hackathon-2026
    # Drop anything already imported from the old copy, so this works without a restart.
    for _name in [m for m in sys.modules if m.startswith("kaai_hackathon")]:
        del sys.modules[_name]

from kaai_hackathon.colab_setup import setup

setup()


In [ ]:
%matplotlib inline
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from kaai_hackathon import PARAM_NAMES, PUBLIC_SUITES
from kaai_hackathon.catalog_io import read_catalog
from kaai_hackathon.splits import (
    example_sims, load_labels, local_split, make_split,
)

DATA_ROOT = Path(os.environ["CAMELS_HACKATHON_DATA"])
PARAMS_ROOT = Path(os.environ.get("CAMELS_HACKATHON_PARAMS", DATA_ROOT))


def catalog_path(suite, sim_id):
    return DATA_ROOT / suite / f"LH_{sim_id}" / "groups_090.hdf5"


print("parameters:", PARAM_NAMES)

## 1. The six parameters

Two are **cosmological** — properties of the universe itself:

| | meaning | why a catalog notices |
|---|---|---|
| $\Omega_m$ | fraction of the energy density that is matter | more matter means more structure, and more massive halos |
| $\sigma_8$ | how clumpy matter is on an 8 Mpc$/h$ scale | sets the amplitude of the fluctuations structure grows from |

Four are **astrophysical** — knobs on how the simulation code treats gas, written as
multipliers on that code's fiducial setting, so $1.0$ means "the default":

| | roughly |
|---|---|
| `A_SN1` | overall energy budget of galactic winds driven by supernovae |
| `A_SN2` | how fast those winds are launched |
| `A_AGN1` | overall energy budget of black-hole feedback |
| `A_AGN2` | the effective scale of black-hole feedback events |

Two warnings about the astrophysical four.

**They are code-specific.** `A_SN1 = 2.0` in IllustrisTNG and `A_SN1 = 2.0` in SIMBA do not
describe the same physical process — each code implements winds differently, and the
multiplier acts on its own fiducial. The cosmological pair is not like this: $\Omega_m$
means the same thing everywhere.

**They act on gas, and gas is what the codes disagree about.** That is why the suites
produced visibly different mass functions in notebook 00.

## 2. How the 1000 simulations sample those parameters

Each suite is a **latin hypercube**: 1000 simulations, every one with a different
combination of all six parameters. It is not a grid and not one-at-a-time — everything
varies at once, which is why you cannot isolate a single parameter by eye.

Rather than take the documented ranges on faith, read them off the tables.

In [ ]:
for suite in PUBLIC_SUITES:
    table = load_labels(PARAMS_ROOT, suite)
    print(f"--- {suite}  {table.shape}")
    for j, name in enumerate(PARAM_NAMES):
        column = table[:, j]
        linear = np.histogram(column, bins=10)[0].std()
        logged = np.histogram(np.log10(column), bins=10)[0].std()
        kind = "log-uniform" if logged < linear else "uniform"
        print(f"    {name:8s} [{column.min():.3f}, {column.max():.3f}]  {kind}")

So the design is:

- $\Omega_m \in [0.1,\,0.5]$ and $\sigma_8 \in [0.6,\,1.0]$, uniform;
- `A_SN1`, `A_AGN1` $\in [0.25,\,4]$ and `A_SN2`, `A_AGN2` $\in [0.5,\,2]$, uniform in
  $\log$ — sensible, since these are multipliers and a factor of 2 up should be as likely
  as a factor of 2 down.

**Except:** Astrid's `A_AGN2` runs over $[0.25,\,4]$, not $[0.5,\,2]$. If you touch the
bonus feedback targets, do not assume the four suites share ranges — check.

Note also how wide $\Omega_m$ is. The real universe is near $0.3$; these simulations run
from $0.1$ to $0.5$. You are not being asked to measure our universe, you are being asked
to read a parameter off a catalog across a deliberately broad prior.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
table = load_labels(PARAMS_ROOT, "IllustrisTNG")
for j, (ax, name) in enumerate(zip(axes.ravel(), PARAM_NAMES)):
    log_scale = j >= 2
    values = np.log10(table[:, j]) if log_scale else table[:, j]
    ax.hist(values, bins=30, color="C0", alpha=0.85)
    ax.set_xlabel(rf"$\log_{{10}}$({name})" if log_scale else name)
    ax.set_ylabel("simulations")
fig.suptitle("IllustrisTNG latin hypercube: flat in the sampled variable", y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# The six parameters really are varied independently of each other.
table = load_labels(PARAMS_ROOT, "IllustrisTNG")
scaled = np.log10(table.copy())
scaled[:, :2] = table[:, :2]                    # the cosmological pair is linear
correlation = np.corrcoef(scaled.T)

fig, ax = plt.subplots(figsize=(5.2, 4.4))
image = ax.imshow(correlation, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(6), PARAM_NAMES, rotation=45, ha="right")
ax.set_yticks(range(6), PARAM_NAMES)
for a in range(6):
    for b in range(6):
        ax.text(b, a, f"{correlation[a, b]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if abs(correlation[a, b]) > 0.5 else "black")
ax.set_title("Correlation between the six parameters")
fig.colorbar(image, ax=ax, shrink=0.8); plt.tight_layout(); plt.show()

Off-diagonal entries are near zero: the design deliberately decorrelates the parameters, so
a model cannot get one right by guessing it from another.

## 3. Attaching a label to a catalog

The table has 1000 rows. **Row $n$ is simulation `LH_n`.** The mapping is positional, so the
one thing you must never do is sort a list of directory names and zip it against the table —
`LH_10` sorts before `LH_2` and your labels silently shift.

Since this is the single mistake that would poison everything downstream, check it rather
than trust it. Notebook 00 noted that a *training* catalog's header still carries `Omega0`,
which gives us an independent copy of the first parameter to check against.

In [ ]:
import h5py

table = load_labels(PARAMS_ROOT, "IllustrisTNG")
print(f"{'sim':>8} {'Omega_m from params':>20} {'Omega0 in the file':>20} {'match':>7}")
for sim_id in example_sims("IllustrisTNG", 5):
    with h5py.File(catalog_path("IllustrisTNG", sim_id), "r") as f:
        from_file = float(np.asarray(f["Header"].attrs["Omega0"]).ravel()[0])
    from_table = table[sim_id, 0]
    print(f"LH_{sim_id:<5d} {from_table:20.5f} {from_file:20.5f} "
          f"{str(abs(from_file - from_table) < 1e-5):>7}")

print(f"\nand the off-by-one that would have been invisible: "
      f"row 8 says Omega_m = {table[8, 0]:.5f}, not {table[7, 0]:.5f}")

In [ ]:
# In practice you will use labels_public.csv, which carries only the public simulations.
split = make_split("IllustrisTNG")
print(f"public       {len(split['public'])} simulations")
print(f"held out     {len(split['private_test'])} simulations you never receive")
print(f"first public {split['public'][:8]}")

## 4. What can a catalog already tell you?

Before building anything, it is worth seeing how much three crude numbers pick up. For a
few hundred simulations, compute:

- **N** — how many galaxies are above the resolution cut,
- **mean $\log M_\star$** — the typical galaxy mass,
- **neighbours** — the mean number of galaxies within 1 cMpc$/h$, a rough clustering measure.

Three numbers, against six parameters.

For the neighbour count, note the tool: `scipy.spatial.cKDTree(pos, boxsize=box)` wraps at
the box edges natively, so it applies the minimum-image convention for you — and it is
roughly 80x faster than writing out the $N \times N$ distance matrix yourself. Worth
remembering; you will want it again when you build anything spatial.

In [ ]:
from scipy.spatial import cKDTree

MSTAR_CUT = 1.3e-2          # 1e10 Msun/h units
N_SIMS = 300
SUITE = "IllustrisTNG"


def summarise(cat, cut=MSTAR_CUT, radius=1000.0):
    mstar = cat.subhalo["SubhaloMassType"][:, 4]
    keep = mstar > cut
    n = int(keep.sum())
    if n < 2:
        return np.array([0.0, 0.0, 0.0])
    pos = np.mod(cat.subhalo["SubhaloPos"][keep].astype(np.float64), cat.box_size)
    # `boxsize` makes the tree wrap, so this IS the minimum-image convention -- and it is
    # about 80x faster than building the full N x N distance matrix by hand.
    tree = cKDTree(pos, boxsize=cat.box_size)
    neighbours = (tree.count_neighbors(tree, radius) - n) / n    # drop the self-pairs
    log_mstar = np.log10(mstar[keep] * 1e10 / 0.6711)        # -> solar masses
    return np.array([n, log_mstar.mean(), neighbours])


summaries = np.array([
    summarise(read_catalog(catalog_path(SUITE, sim_id), group_fields=[],
                           subhalo_fields=["SubhaloPos", "SubhaloMassType"]))
    for sim_id in example_sims(SUITE, N_SIMS)
])
labels = load_labels(PARAMS_ROOT, SUITE)[example_sims(SUITE, N_SIMS)]
SUMMARY_NAMES = ["N galaxies", r"mean $\log_{10}(M_\star/M_\odot)$",
                 "neighbours < 1 Mpc/h"]
print("summaries:", summaries.shape, " labels:", labels.shape)

In [ ]:
from scipy.stats import spearmanr

rho = np.array([[spearmanr(summaries[:, s], labels[:, p]).statistic
                 for p in range(6)] for s in range(3)])

fig, ax = plt.subplots(figsize=(7.2, 3.2))
image = ax.imshow(rho, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(6), PARAM_NAMES, rotation=30, ha="right")
ax.set_yticks(range(3), SUMMARY_NAMES)
for s in range(3):
    for p in range(6):
        ax.text(p, s, f"{rho[s, p]:+.2f}", ha="center", va="center", fontsize=9,
                color="white" if abs(rho[s, p]) > 0.55 else "black")
ax.set_title(f"Rank correlation, {N_SIMS} {SUITE} simulations")
fig.colorbar(image, ax=ax, shrink=0.9, label="Spearman rho")
plt.tight_layout(); plt.show()

In [ ]:
# The two strongest cells, drawn out.
flat = np.dstack(np.meshgrid(range(3), range(6), indexing="ij")).reshape(-1, 2)
best = flat[np.argsort(-np.abs(rho.ravel()))[:2]]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (s, p) in zip(axes, best):
    ax.scatter(labels[:, p], summaries[:, s], s=14, alpha=0.7)
    ax.set_xlabel(PARAM_NAMES[p]); ax.set_ylabel(SUMMARY_NAMES[s])
    ax.set_title(rf"$\rho$ = {rho[s, p]:+.2f}")
    if p >= 2:
        ax.set_xscale("log")
        ticks = [0.25, 0.5, 1.0, 2.0, 4.0]
        ticks = [v for v in ticks
                 if labels[:, p].min() * 0.98 <= v <= labels[:, p].max() * 1.02]
        ax.set_xticks(ticks, [str(v) for v in ticks])
        ax.minorticks_off()
plt.tight_layout(); plt.show()

Read this as a floor, not a ceiling. Three hand-picked numbers, one suite, a rank
correlation — that is the crudest possible probe. A real model sees every galaxy, every
column, and the full spatial arrangement, and can combine them in ways a single scalar
cannot. Where the correlation here is weak, the honest statement is "these three numbers do
not capture it", not "the information is absent".

What the table *is* good for: it tells you which quantities in a catalog visibly move when
you turn each knob, which is useful when you are choosing what to feed a model.

## 5. The task

**Predict** $\Omega_m$ and $\sigma_8$ from a catalog. The four astrophysical parameters are
an optional bonus track, scored separately.

**Your data.** 900 simulations per suite for IllustrisTNG, SIMBA and Astrid — 2700
catalogs with full labels. How you split them into training and validation is up to you.
Splitting **by simulation** matters: two galaxies from the same simulation are not
independent samples, so putting some of a simulation's galaxies in train and the rest in
validation will flatter your validation score.

**Your test.** 100 held-out simulations per suite, which you never see, plus a fourth
simulation code you have never trained on.

**The conditions.** Every catalog you are scored on is put through one of these first:

In [ ]:
from kaai_hackathon.conditions import PUBLISHED_CONDITIONS

print(f"{'condition':16s} {'tier':6s} {'operation':22s} settings")
for spec in PUBLISHED_CONDITIONS:
    print(f"{spec.name:16s} {spec.tier:6s} {spec.op:22s} {spec.params or ''}")

Tier `S` are **symmetries**: shifting the box origin, rotating it by a right angle,
reordering the rows. None of them changes the physics at all, so a model built the right
way should score identically before and after. Tier `C` are **corruptions**, which really
do remove information. On top of those sits the held-out simulation code.

**The score** is $R^2$ across the held-out simulations, reported for each condition and
each target separately — never averaged into one number. $R^2 = 1$ is perfect, $R^2 = 0$ is
no better than always predicting the mean, and negative is worse than that. Reporting per
condition is deliberate: a model that is a little less accurate but does not fall apart
under a shift is the more interesting result.

**What you hand in** is a `predict.py` exposing two functions, not a file of predictions:

```python
def load_model(model_dir: str) -> object: ...
def predict(model, catalog_path: str) -> dict:
    return {"Omega_m": ..., "sigma_8": ...}
```

Notebook 05 walks through it, and a working example ships with the kit.

## What you should take away

1. Six parameters: two cosmological (universal meaning), four astrophysical (code-specific
   multipliers on that code's own defaults).
2. Sampled as a latin hypercube — all six vary at once and are mutually decorrelated;
   $\Omega_m$, $\sigma_8$ uniform, the four `A_*` log-uniform, and Astrid's `A_AGN2` has a
   wider range than the others.
3. Row $n$ of the parameter table is `LH_n`. Never zip sorted directory names against it.
4. Split by simulation, never by galaxy.
5. You are scored per condition and per target, on simulations and a simulation code you
   have never seen.

**Next:** notebook `02` turns a catalog — a set of thousands of galaxies — into something a
regressor can actually accept.